# 03 — Statystyka hydrologiczna

Ten notebook realizuje analize statystyczna oczyszczonych danych przeplywu:

1. **Przeplywy charakterystyczne** — NNQ, SNQ, SSQ, SWQ, WWQ (notacja polska)
2. **Krzywa sum czasow przeplywu (FDC)** — prawdopodobienstwo przekroczenia
3. **Wzorce miesieczne** — zmiennosc sezonowa
4. **Trendy roczne** — zmiany wieloletnie
5. **Rozklad przeplywu** — histogram i dopasowanie log-normalne

Te wyniki sa niezbedne do oceny potencjalu energetycznego MEW (notebook 04).

---

## Konfiguracja

**Prompt do LLM:**
> *"Napisz kod konfiguracji notebooka: załaduj moduł src/imgw (load_processed) i src/hydro_stats (flow_duration_curve, characteristic_flows, monthly_stats, annual_stats)."*

**Użyte moduły/funkcje:** `src.imgw`: load_processed; `src.hydro_stats`: flow_duration_curve, characteristic_flows, monthly_stats, annual_stats

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.imgw import load_processed
from src.hydro_stats import (
    flow_duration_curve,
    characteristic_flows,
    monthly_stats,
    annual_stats,
)

pd.set_option('display.max_columns', 15)
print('Moduly zaladowane.')

### Moduł `src/hydro_stats.py` — statystyka hydrologiczna

**Prompt do LLM tworzący ten moduł:**
> *"Napisz modul Pythona src/hydro_stats.py do analizy statystycznej danych hydrologicznych.
> Moduł powinien:
> 1. Obliczać krzywą sum czasów przepływu (FDC) — sortowanie malejąco, procent przekroczenia.
> 2. Obliczać przepływy charakterystyczne wg polskiej notacji: NNQ, SNQ, SSQ, SWQ, WWQ, Q50, Q90, Q95.
> 3. Obliczać statystyki miesięczne i roczne (mean, min, max, std).
> UWAGA: Q90 to przeplyw przekraczany przez 90% czasu = 10. percentyl (nie 90. percentyl!)."*

**Wynik:** Moduł `src/hydro_stats.py` z funkcjami:
- `flow_duration_curve(df, station_id)` — DataFrame z discharge i exceedance_pct
- `characteristic_flows(df, station_id)` — Series z NNQ, SNQ, SSQ, SWQ, WWQ, Q50, Q90, Q95
- `monthly_stats(df, station_id)` / `annual_stats(df, station_id)` — statystyki okresowe

**Prompt do LLM:**
> *"Napisz kod ktory uzyje load_processed() z modułu src/imgw do wczytania danych z daily_hydro_clean.parquet. Wybierz pierwszą stację jako główną i wyświetl jej nazwę i rzekę."*

**Użyte moduły/funkcje:** `src.imgw.load_processed()` — DataFrame z kolumnami: station_id, station_name, river_name, water_level_cm, discharge_m3s, date

In [ ]:
# Wczytaj oczyszczone dane
df = load_processed('../data/processed/daily_hydro_clean.parquet')
print(f'Wczytano {len(df)} wierszy')

# Uzyj stacji glownej do analizy
PRIMARY_STATION = df['station_id'].unique()[0]
name = df.loc[df['station_id'] == PRIMARY_STATION, 'station_name'].iloc[0]
river = df.loc[df['station_id'] == PRIMARY_STATION, 'river_name'].iloc[0]
print(f'Stacja glowna: {name} ({PRIMARY_STATION}) na rzece {river}')

---
## Krok 1: Przeplywy charakterystyczne

Polska notacja hydrologiczna:

| Symbol | Nazwa | Znaczenie |
|--------|-------|-----------|
| NNQ | Najnizszy z najnizszych | Minimum absolutne |
| SNQ | Sredni z najnizszych | Srednia z rocznych minimow |
| SSQ | Sredni ze srednich | Przeplyw sredni |
| SWQ | Sredni z najwyzszych | Srednia z rocznych maksimow |
| WWQ | Najwyzszy z najwyzszych | Maksimum absolutne |
| Q50 | — | Mediana przeplywu |
| Q90 | — | Przeplyw przekraczany przez 90% czasu |
| Q95 | — | Przeplyw przekraczany przez 95% czasu |

**Prompt do LLM:**
> *"Napisz kod ktory użyje funkcji characteristic_flows() z modułu src/hydro_stats do obliczenia przepływów charakterystycznych. Wyświetl wyniki w czytelnej formie z jednostkami m3/s."*

**Użyte moduły/funkcje:** `src.hydro_stats.characteristic_flows()` — min/max absolutne, srednie roczne min/max, percentyle

In [ ]:
# Oblicz przeplywy charakterystyczne
chars = characteristic_flows(df, PRIMARY_STATION)
print(f'Przeplywy charakterystyczne dla {name} ({river}):')
print('=' * 40)
for key, val in chars.items():
    if key.startswith('n_'):
        print(f'  {key:20s}: {int(val)}')
    else:
        print(f'  {key:20s}: {val:10.3f} m3/s')

---
## Krok 2: Krzywa sum czasow przeplywu (FDC)

Krzywa FDC pokazuje, przez jaki procent czasu dany przeplyw jest **przekraczany**.
Jest to fundamentalne narzedzie do projektowania MEW — mowi nam ile wody jest dostepne i jak czesto.

Kluczowe odczyty:
- **Q50** — przeplyw medianowy (przekraczany przez 50% czasu)
- **Q90** — przeplyw pewny (przekraczany przez 90% czasu, uzywany do mocy gwarantowanej)
- **Q95** — przeplyw niski (przekraczany przez 95% czasu)

**Prompt do LLM:**
> *"Napisz kod ktory użyje funkcji flow_duration_curve() z modułu src/hydro_stats i narysuje FDC używając plotly graph_objects. Os Y logarytmiczna. Dodaj poziome linie przerywane dla Q50, Q90, Q95 z adnotacjami."*

**Użyte moduły/funkcje:** `src.hydro_stats.flow_duration_curve()` — sortuje malejaco, oblicza procent przekroczenia; `plotly.graph_objects` z add_hline

In [ ]:
fdc = flow_duration_curve(df, PRIMARY_STATION)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fdc['exceedance_pct'], y=fdc['discharge_m3s'],
    mode='lines', name='FDC',
    line=dict(color='royalblue', width=2),
))

# Dodaj linie przeplywow charakterystycznych
for q_name, q_pct, color in [('Q50', 50, 'green'), ('Q90', 90, 'orange'), ('Q95', 95, 'red')]:
    q_val = chars[q_name]
    fig.add_hline(y=q_val, line_dash='dash', line_color=color,
                  annotation_text=f'{q_name} = {q_val:.2f} m3/s')

fig.update_layout(
    title=f'Krzywa sum czasow przeplywu - {name} ({river})',
    xaxis_title='Prawdopodobienstwo przekroczenia [%]',
    yaxis_title='Q [m3/s]',
    yaxis_type='log',
    height=500,
    hovermode='x unified',
)
fig.show()

**Prompt do LLM:**
> *"Napisz kod rysujący tę samą FDC ale w skali liniowej, z wypełnieniem pod krzywą (fill='tozeroy' w plotly Scatter)."*

**Użyte moduły/funkcje:** `plotly.graph_objects.Scatter(fill='tozeroy')` — wypelnienie do osi X

In [ ]:
# FDC w skali liniowej
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fdc['exceedance_pct'], y=fdc['discharge_m3s'],
    mode='lines', name='FDC', fill='tozeroy',
    line=dict(color='royalblue', width=2),
))
fig.update_layout(
    title=f'Krzywa sum czasow przeplywu (skala liniowa) - {name}',
    xaxis_title='Prawdopodobienstwo przekroczenia [%]',
    yaxis_title='Q [m3/s]',
    height=450,
)
fig.show()

---
## Krok 3: Wzorce miesieczne

Sezonowa zmiennosc przeplywu jest wazna dla zrozumienia kiedy dostepne jest najwiecej/najmniej wody.

**Prompt do LLM:**
> *"Napisz kod ktory użyje funkcji monthly_stats() z modułu src/hydro_stats i wyświetli tabelkę statystyk miesięcznych."*

**Użyte moduły/funkcje:** `src.hydro_stats.monthly_stats()` — groupby month, agg mean/min/max/std

In [ ]:
mstats = monthly_stats(df, PRIMARY_STATION)
print(f'Statystyki miesieczne dla {name}:')
mstats

**Prompt do LLM:**
> *"Napisz kod ktory narysuje wykres miesięczny używając plotly graph_objects: słupki Bar ze średnim przepływem i error_y (std), plus linie Scatter dla max i min. Nazwy miesięcy po polsku."*

**Użyte moduły/funkcje:** `plotly.graph_objects.Bar()` z error_y + `Scatter()` dla min/max

In [ ]:
month_names = ['Sty', 'Lut', 'Mar', 'Kwi', 'Maj', 'Cze',
               'Lip', 'Sie', 'Wrz', 'Paz', 'Lis', 'Gru']

fig = go.Figure()
fig.add_trace(go.Bar(
    x=month_names, y=mstats['mean'],
    name='Sredni Q',
    error_y=dict(type='data', array=mstats['std'], visible=True),
    marker_color='royalblue',
))
fig.add_trace(go.Scatter(
    x=month_names, y=mstats['max'],
    name='Maks Q', mode='markers+lines',
    line=dict(color='red', dash='dot'),
))
fig.add_trace(go.Scatter(
    x=month_names, y=mstats['min'],
    name='Min Q', mode='markers+lines',
    line=dict(color='green', dash='dot'),
))
fig.update_layout(
    title=f'Miesieczny rozklad przeplywu - {name}',
    yaxis_title='Q [m3/s]', height=450,
)
fig.show()

---
## Krok 4: Trendy roczne

**Prompt do LLM:**
> *"Napisz kod ktory uzyje annual_stats() z modułu src/hydro_stats i narysuje wykres roczny: Bar (średni Q), Scatter (min/max), pozioma linia SSQ z adnotacją."*

**Użyte moduły/funkcje:** `src.hydro_stats.annual_stats()` + `plotly.graph_objects` — Bar + Scatter + add_hline

In [ ]:
astats = annual_stats(df, PRIMARY_STATION)

fig = go.Figure()
fig.add_trace(go.Bar(x=astats.index, y=astats['mean'], name='Sredni Q', marker_color='royalblue'))
fig.add_trace(go.Scatter(x=astats.index, y=astats['max'], name='Maks Q', mode='markers', marker=dict(color='red', size=5)))
fig.add_trace(go.Scatter(x=astats.index, y=astats['min'], name='Min Q', mode='markers', marker=dict(color='green', size=5)))

# Dodaj linie SSQ
fig.add_hline(y=chars['SSQ'], line_dash='dash', line_color='black',
              annotation_text=f'SSQ = {chars["SSQ"]:.2f} m3/s')

fig.update_layout(
    title=f'Przeplyw roczny - {name}',
    xaxis_title='Rok', yaxis_title='Q [m3/s]',
    height=450,
)
fig.show()

---
## Krok 5: Rozklad przeplywu

Przeplyw dobowy zazwyczaj ma **rozklad log-normalny**. Sprawdzmy to.

**Prompt do LLM:**
> *"Napisz kod z plotly make_subplots ktory narysuje 2 histogramy obok siebie: przeplyw Q i ln(Q). Użyj np.log() do transformacji."*

**Użyte moduły/funkcje:** `plotly.subplots.make_subplots()` — 2 histogramy, np.log() dla transformacji

In [ ]:
sdf = df[df['station_id'] == PRIMARY_STATION]['discharge_m3s'].dropna()

fig = make_subplots(rows=1, cols=2, subplot_titles=['Histogram przeplywu', 'Histogram ln(przeplywu)'])

fig.add_trace(
    go.Histogram(x=sdf, nbinsx=80, name='Q', marker_color='royalblue'),
    row=1, col=1
)
fig.add_trace(
    go.Histogram(x=np.log(sdf), nbinsx=80, name='ln(Q)', marker_color='darkorange'),
    row=1, col=2
)
fig.update_xaxes(title_text='Q [m3/s]', row=1, col=1)
fig.update_xaxes(title_text='ln(Q)', row=1, col=2)
fig.update_layout(title=f'Rozklad przeplywu - {name}', height=400, showlegend=False)
fig.show()

---
## Podsumowanie

Kluczowe wyniki dla projektu MEW:

| Parametr | Wartosc | Zastosowanie |
|----------|---------|-------------|
| SSQ | Przeplyw sredni | Ogolne wymiarowanie |
| Q90 | Przeplyw pewny | Moc gwarantowana |
| Q95 | Przeplyw niski | Minimalna eksploatacja |
| FDC | Pelna krzywa | Szacowanie produkcji energii |
| Wzorzec miesieczny | Zmiennosc sezonowa | Planowanie eksploatacji |

**Dalej:** `04_power_potential.ipynb` — obliczanie mocy i rocznej produkcji energii na podstawie FDC